# Task #58 — Giảm scale_pos_weight, tinh ngưỡng lại, so sánh F1 (Story #10)

**Giả thuyết** (mục 4 #5, gợi ý từ advisor): `scale_pos_weight=11.3233` (Task #47) được tính bằng đúng tỉ lệ mất cân bằng thật (n_âm/n_dương) — cách làm chuẩn để cân bằng **recall giữa 2 lớp**, không phải để tối ưu **F1 trên lớp dương**. Trọng số quá cao có thể đẩy xác suất dự đoán lệch mạnh về phía "trễ", khiến đường cong Precision-Recall bị kéo lệch bất lợi cho F1.

**Cách kiểm chứng**: quét `scale_pos_weight` qua nhiều giá trị nhỏ hơn 11.3233 (kể cả 1.0 = không trọng số), huấn luyện lại XGBoost trên tập train đã có đặc trưng khoảng cách (Task #57), đo **F1 tối đa qua tinh ngưỡng** + PR-AUC trên test cho từng giá trị. Nếu giả thuyết đúng, sẽ có 1 giá trị < 11.3233 cho F1 tối đa cao hơn kết quả baseline (0.351, Task #57).

**Phạm vi**: chỉ XGBoost (model dẫn đầu, mục 4 #4) — không lặp lại cho Random Forest/Logistic Regression để giữ thí nghiệm gọn, đúng tinh thần "đòn bẩy rẻ, kiểm tra nhanh".

In [1]:
import pandas as pd
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from xgboost import XGBClassifier

train_df = pd.read_csv("../data/processed/orders_features_train.csv", low_memory=False)
test_df = pd.read_csv("../data/processed/orders_features_test.csv", low_memory=False)

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for df in (train_df, test_df):
    df["is_delayed"] = df["is_delayed"].astype(bool)
    for col in bool_cols:
        df[col] = df[col].astype("boolean")

X_train = train_df.drop(columns=["order_id", "is_delayed"])
y_train = train_df["is_delayed"].astype(int)
X_test = test_df.drop(columns=["order_id", "is_delayed"])
y_test = test_df["is_delayed"].astype(int)

print("X_train:", X_train.shape, " X_test:", X_test.shape)

X_train: (77156, 76)  X_test: (19289, 76)


## 1. Quét scale_pos_weight, đo F1 tối đa qua tinh ngưỡng + PR-AUC cho từng giá trị

In [2]:
weights_to_try = [1.0, 2.0, 4.0, 6.0, 8.0, 11.3233]

sweep_rows = []
for w in weights_to_try:
    model = XGBClassifier(scale_pos_weight=w, random_state=42, n_jobs=-1, eval_metric="logloss")
    model.fit(X_train, y_train)

    y_score = model.predict_proba(X_test)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_test, y_score)
    f1_per_threshold = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = f1_per_threshold.argmax()

    sweep_rows.append({
        "scale_pos_weight": w,
        "f1_at_default_0.5": f1_score(y_test, (y_score >= 0.5).astype(int)),
        "max_f1_via_threshold": f1_per_threshold[best_idx],
        "best_threshold": thresholds[best_idx],
        "average_precision_pr_auc": average_precision_score(y_test, y_score),
    })
    print(f"scale_pos_weight={w}: max_f1_via_threshold={f1_per_threshold[best_idx]:.4f}, "
          f"PR-AUC={average_precision_score(y_test, y_score):.4f}")

sweep_df = pd.DataFrame(sweep_rows).sort_values("max_f1_via_threshold", ascending=False).reset_index(drop=True)
sweep_df

scale_pos_weight=1.0: max_f1_via_threshold=0.3482, PR-AUC=0.2859


scale_pos_weight=2.0: max_f1_via_threshold=0.3512, PR-AUC=0.2966


scale_pos_weight=4.0: max_f1_via_threshold=0.3462, PR-AUC=0.2860


scale_pos_weight=6.0: max_f1_via_threshold=0.3530, PR-AUC=0.2931


scale_pos_weight=8.0: max_f1_via_threshold=0.3498, PR-AUC=0.2947


scale_pos_weight=11.3233: max_f1_via_threshold=0.3510, PR-AUC=0.2855


,scale_pos_weight,f1_at_default_0.5,max_f1_via_threshold,best_threshold,average_precision_pr_auc
0,6.0000,0.349623,0.352973,0.539085,0.293124
1,2.0000,0.268893,0.351183,0.273853,0.296562
2,11.3233,0.317019,0.350995,0.637262,0.285490
3,8.0000,0.341189,0.349761,0.553290,0.294709
4,1.0000,0.155663,0.348150,0.180205,0.285911
5,4.0000,0.336342,0.346154,0.421670,0.286009


## 2. Lưu kết quả, so sánh với baseline (scale_pos_weight=11.3233, dòng cuối trong bảng quét ở trên)

In [3]:
sweep_df.to_csv("../models/class_weight_sweep.csv", index=False)

best_row = sweep_df.iloc[0]
baseline_row = sweep_df[sweep_df["scale_pos_weight"] == 11.3233].iloc[0]
print(f"Gia tri scale_pos_weight tot nhat: {best_row['scale_pos_weight']}")
print(f"max_f1_via_threshold tot nhat: {best_row['max_f1_via_threshold']:.4f} "
      f"(baseline scale_pos_weight=11.3233: {baseline_row['max_f1_via_threshold']:.4f}, "
      f"chenh lech: {best_row['max_f1_via_threshold'] - baseline_row['max_f1_via_threshold']:+.4f})")
print("Da luu models/class_weight_sweep.csv")

Gia tri scale_pos_weight tot nhat: 6.0
max_f1_via_threshold tot nhat: 0.3530 (baseline scale_pos_weight=11.3233: 0.3510, chenh lech: +0.0020)
Da luu models/class_weight_sweep.csv
